# Phase 1 Smoke Tests

This notebook is for small, local checks of the Phase 1 synthetic refactor.

It covers:
- tiny manual network creation with the extracted core imports
- a very small geometry-first generation and width-sampling run
- exact reproduction of the preserved 7-record sensitivity recipe file
- explicit inspection of the shared single-edge control artifact
- Plotly inspection of the 7 preserved sensitivity networks


In [7]:
from pathlib import Path
import sys
import shutil
import gzip
import json
import pandas as pd


def find_synthetic_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / "synthetic_runs",
        Path.cwd().parent / "synthetic_runs",
        Path("/Users/6256481/Code/river-hierarchy/synthetic_runs"),
    ]
    for candidate in candidates:
        if (candidate / "src" / "synthetic_runs").exists():
            return candidate.resolve()
    raise RuntimeError("Could not locate synthetic_runs root.")


ROOT = find_synthetic_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

OUT = ROOT / "outputs" / "test_phase_1"
OUT.mkdir(parents=True, exist_ok=True)

SENSITIVITY_REF = Path(
    "/Volumes/PhD/river_hierarchy/output/synthetic_network/"
    "synthetic_run_sensitivity/networks_sensitivity.jsonl.gz"
)

print("ROOT:", ROOT)
print("OUT:", OUT)
print("SENSITIVITY_REF exists:", SENSITIVITY_REF.exists())


ROOT: /Users/6256481/Code/river-hierarchy/synthetic_runs
OUT: /Users/6256481/Code/river-hierarchy/synthetic_runs/outputs/test_phase_1
SENSITIVITY_REF exists: True


In [8]:
from synthetic_runs.core import Params, RiverNetworkNX, k_stats_from_graph
from synthetic_runs.runners.sensitivity_recipes import (
    build_default_sensitivity_recipes,
    write_recipes,
    validate_recipes,
    main as sensitivity_main,
)
from synthetic_runs.runners import load_single_edge_control, resolve_single_edge_control
from synthetic_runs.analysis import plot_recipe_grid, recipe_summary_frame

import synthetic_geometric_enumeration as geom_enum
import synthetic_width_perc_splits as width_splits


## 1. Tiny manual network creation

This checks that the extracted core imports work and that we can still build
a very small synthetic network through the preserved `RiverNetworkNX`
compatibility export.


In [11]:
small_p = Params(
    L=3000,
    W_total=40.0,
    xs=500,
    xe=2500,
    jump=500,
    max_breaks=3,
    min_width=10.0,
    width_step=10,
    x_stability=0.3,
)

base_net = RiverNetworkNX(small_p)
base_net.instantiate_corridor(20.0, 20.0)
base_stats = k_stats_from_graph(base_net.G, small_p.x_stability)

loop_net = RiverNetworkNX(small_p)
loop_net.instantiate_corridor(20.0, 20.0)
loop_net.add_loop(branch="A", xb=1000.0, xr=1500.0, W1=10.0, W2=10.0, replace_corridor=True)
loop_stats = k_stats_from_graph(loop_net.G, small_p.x_stability)

print("base nodes/edges:", len(base_net.G.nodes), len(base_net.G.edges))
print("base stats:", base_stats)
print("loop nodes/edges:", len(loop_net.G.nodes), len(loop_net.G.edges))
print("loop path count:", loop_net.count_paths())
print("loop stats:", loop_stats)

base_net.to_recipe()


base nodes/edges: 4 4
base stats: {'k_sum': 30746.563544555756, 'k_min': 2091.7155995706853, 'k_max': 13281.566172707193, 'k_mean': 7686.640886138939, 'k_ratio': 6.349604207872797, 'ebi_mean': np.float64(2.0), 'ebi_max': np.float64(2.0), 'admissible': False}
loop nodes/edges: 6 7
loop path count: 3
loop stats: {'k_sum': 37967.75805854928, 'k_min': 2091.7155995706853, 'k_max': 13281.566172707193, 'k_mean': 5423.965436935612, 'k_ratio': 6.349604207872797, 'ebi_mean': np.float64(2.2761423749153966), 'ebi_max': np.float64(2.8284271247461903), 'admissible': False}


{'meta': {'L': 3000,
  'W_total': 40.0,
  'xs': 500,
  'xe': 2500,
  'jump': 500,
  'min_width': 10.0,
  'width_step': 10,
  'x_stability': 0.3,
  'max_breaks': 3,
  'Y0': 1.0,
  'amp_corr': 1.5,
  'amp_loop': 0.7},
 'initial_split': {'WA': 20.0, 'WB': 20.0},
 'breaks': []}

## 2. Tiny geometry-first generation and width sampling

This is intentionally much smaller than the legacy production setup. It writes
to `synthetic_runs/outputs/test_phase_1/`.


In [10]:
geom_out = OUT / "geometry_small"
sampled_out = OUT / "sampled_small"

for path in [geom_out, sampled_out]:
    if path.exists():
        shutil.rmtree(path)

tiny_p = Params(
    L=3000,
    W_total=40.0,
    xs=500,
    xe=2500,
    jump=500,
    max_breaks=1,
    min_width=10.0,
    width_step=10,
    x_stability=0.3,
)

geom_paths = geom_enum.enumerate_geometric_recipes_streamed(
    tiny_p,
    out_dir=geom_out,
    min_span=tiny_p.jump,
)

print(geom_paths)
pd.DataFrame([geom_paths])


{'meta': '/Users/6256481/Code/river-hierarchy/synthetic_runs/outputs/test_phase_1/geometry_small/run_meta_geometry.json', 'recipes': '/Users/6256481/Code/river-hierarchy/synthetic_runs/outputs/test_phase_1/geometry_small/geometry_recipes.jsonl.gz', 'n_written': 13, 'n_tested_nodes': 13, 'n_candidates': 12, 'dedup_sets_size': 13}


,meta,recipes,n_written,n_tested_nodes,n_candidates,dedup_sets_size
0,/Users/6256481/Code/river-hierarchy/synthetic_...,/Users/6256481/Code/river-hierarchy/synthetic_...,13,13,12,13


In [12]:
sample_paths = width_splits.sample_realized_networks_from_geometry(
    params_json=geom_out / "run_meta_geometry.json",
    geometry_recipes_gz=geom_out / "geometry_recipes.jsonl.gz",
    out_dir=sampled_out,
    n_samples=2,
    seed=42,
    max_attempts_per_sample=50,
    filter_k_admissible=False,
    summary_format="csv",
    write_edges=True,
)

print(sample_paths)
pd.read_csv(sample_paths["summary"]).head()


{'params_used': '/Users/6256481/Code/river-hierarchy/synthetic_runs/outputs/test_phase_1/geometry_small/run_meta_geometry.json', 'geometry_recipes': '/Users/6256481/Code/river-hierarchy/synthetic_runs/outputs/test_phase_1/geometry_small/geometry_recipes.jsonl.gz', 'networks': '/Users/6256481/Code/river-hierarchy/synthetic_runs/outputs/test_phase_1/sampled_small/networks.jsonl.gz', 'summary': '/Users/6256481/Code/river-hierarchy/synthetic_runs/outputs/test_phase_1/sampled_small/summary.csv', 'edges': '/Users/6256481/Code/river-hierarchy/synthetic_runs/outputs/test_phase_1/sampled_small/edges.csv', 'failures': '/Users/6256481/Code/river-hierarchy/synthetic_runs/outputs/test_phase_1/sampled_small/failures.csv', 'n_realized': 26}


,network_id,geometry_id,sample_id,is_benchmark,n_breaks,n_paths,WA0,WB0,k_sum,k_min,k_max,k_mean,k_ratio,ebi_mean,ebi_max,admissible,ebi_min,bi
0,0,0,0,True,0,2,20.0,20.0,30746.563545,2091.7156,13281.566173,7686.640886,6.349604,2.000000,2.000000,False,2.000000,2.000000
1,1,0,1,False,0,2,24.0,16.0,31356.798597,2091.7156,15411.892310,7839.199649,7.368063,1.960132,1.960132,False,1.960132,2.000000
2,2,1,0,True,1,3,20.0,20.0,40295.643930,2091.7156,15812.379086,5756.520561,7.559526,2.194397,2.828427,False,1.754765,2.333333
3,3,1,1,False,1,3,24.0,16.0,38806.331629,2091.7156,14002.633446,5543.761661,6.694330,2.257720,2.971004,False,1.842023,2.333333
4,4,2,0,True,1,3,20.0,20.0,46352.895508,2091.7156,15812.379086,6621.842215,7.559526,2.194397,2.828427,False,1.754765,2.333333


## 3. Sensitivity recipe builder

This tests both the pure Python helper and the direct CLI-style entrypoint.


In [4]:
sensitivity_out = OUT / "networks_sensitivity_phase1.jsonl.gz"

recipes = build_default_sensitivity_recipes()
write_recipes(recipes, sensitivity_out)

print("recipe count:", len(recipes))
print("matches preserved fixture:", validate_recipes(recipes, SENSITIVITY_REF))

with gzip.open(sensitivity_out, "rt", encoding="utf-8") as f:
    first_recipe = json.loads(next(f))

first_recipe


recipe count: 7
matches preserved fixture: True


{'meta': {'L': 12000,
  'W_total': 650.0,
  'xs': 3000,
  'xe': 9000,
  'jump': 500,
  'min_width': 10.0,
  'width_step': 10,
  'x_stability': 0.3,
  'max_breaks': 5,
  'Y0': 1.0,
  'amp_corr': 1.5,
  'amp_loop': 0.7},
 'initial_split': {'WA': 520.0, 'WB': 130.0},
 'breaks': [{'kind': 'loop',
   'from_branch': 'A',
   'to_branch': 'A',
   'xb': 5000.0,
   'xr': 7000.0,
   'w1': 416.0,
   'w2': 104.0,
   'replace_corridor': True}],
 'geometry_id': 1000001,
 'sample_id': 0,
 'sample_mode': 'benchmark',
 'sample_plan': {'initial': {'ratio': [0.5, 0.5], 'flip': False},
  'per_break': [{'ratio': [], 'flip': False}]}}

In [5]:
cli_out = OUT / "networks_sensitivity_phase1_cli.jsonl.gz"

_ = sensitivity_main([
    "--out", str(cli_out),
    "--validate-against", str(SENSITIVITY_REF),
])

print("CLI output exists:", cli_out.exists())


wrote /Users/6256481/Code/river-hierarchy/synthetic_runs/outputs/test_phase_1/networks_sensitivity_phase1_cli.jsonl.gz rows 7
validated against /Volumes/PhD/river_hierarchy/output/synthetic_network/synthetic_run_sensitivity/networks_sensitivity.jsonl.gz
CLI output exists: True


## 4. Explicit single-edge control artifact

The 7 sensitivity recipes remain the structural set. The extra single-edge
baseline is now a separate shared control artifact for sampled and sensitivity
runs.


In [6]:
single_edge_spec, single_edge_control_path = load_single_edge_control()
single_edge_resolved = resolve_single_edge_control(single_edge_spec, first_recipe["meta"])

print("control path:", single_edge_control_path)
single_edge_resolved


control path: /Users/6256481/Code/river-hierarchy/synthetic_runs/configs/single_edge_control.json


{'control_type': 'single_edge',
 'network_id': -1,
 'geometry_id': -1,
 'sample_type': 'single_edge',
 'network_type': 'single_edge',
 'length_m': 12000.0,
 'width_m': 650.0}

## 5. Plotly inspection of the 7 sensitivity recipes

This gives a quick visual check that the preserved sensitivity file still
contains the expected 2 loop, 3 cross, and 2 no-break cases.


In [8]:
recipe_summary_frame(SENSITIVITY_REF, network_ids=range(7))


,network_id,geometry_id,sample_id,sample_mode,WA,WB,n_breaks,break_summary
0,0,1000001,0,benchmark,520.0,130.0,1,"loop A 5000-7000 w=(416,104)"
1,1,1000002,0,benchmark,520.0,130.0,1,"loop A 5000-7000 w=(260,260)"
2,2,1000003,0,benchmark,520.0,130.0,1,cross A->B 5000->7000 w=416
3,3,1000004,0,benchmark,520.0,130.0,1,cross A->B 5000->7000 w=260
4,4,1000005,0,benchmark,520.0,130.0,1,cross A->B 5000->7000 w=104
5,5,1000006,0,benchmark,520.0,130.0,0,no-break
6,6,1000007,0,benchmark,325.0,325.0,0,no-break


In [9]:
fig = plot_recipe_grid(
    SENSITIVITY_REF,
    network_ids=range(7),
    cols=3,
    title="Sensitivity recipe set",
)
fig


## Direct script usage

You can also run the builder directly from a shell:

```bash
python synthetic_runs/src/synthetic_runs/runners/sensitivity_recipes.py \
  --out /tmp/networks_sensitivity_phase1.jsonl.gz \
  --validate-against /Volumes/PhD/river_hierarchy/output/synthetic_network/synthetic_run_sensitivity/networks_sensitivity.jsonl.gz
```
